# 02. 실습: 의미 벡터와 간단한 이상 탐지기

목표: BERT 대신 재현 가능한 toy semantic vector를 만들고, 정상/이상 prototype classifier로 로그 윈도우를 분류합니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. Deterministic token embedding

실제 NeuralLog는 BERT hidden state 평균을 사용합니다. 여기서는 `hashlib`로 항상 같은 작은 벡터를 만들어 전체 흐름만 재현합니다.

In [ ]:
import hashlib
import math
import re
import string


def token_vector(token, dim=8):
    digest = hashlib.sha256(token.encode("utf-8")).digest()
    values = []
    for i in range(dim):
        # 0..255를 -1..1 범위로 바꿉니다.
        values.append((digest[i] / 127.5) - 1.0)
    return values


def mean_vector(vectors):
    if not vectors:
        return [0.0] * 8
    return [sum(vector[i] for vector in vectors) / len(vectors) for i in range(len(vectors[0]))]


def message_vector(message):
    tokens = re.findall(r"[a-z]+", message.lower())
    return mean_vector([token_vector(token) for token in tokens])


for word in ["disk", "timeout", "login"]:
    print(word, [round(value, 3) for value in token_vector(word)[:4]])

## 2. 작은 학습/평가 데이터 만들기

각 윈도우는 여러 로그 메시지 벡터의 평균으로 표현합니다. 실제 NeuralLog는 이 부분을 Transformer Encoder가 처리합니다.

In [ ]:
train_windows = [
    (["node disk read completed", "network link recovered", "cache warmup completed"], 0),
    (["user login accepted", "service heartbeat ok", "job scheduled normally"], 0),
    (["disk read timeout", "io queue stalled", "kernel reports failure"], 1),
    (["kernel panic reboot required", "service unavailable", "failure reported"], 1),
]

test_windows = [
    (["disk read completed", "network link recovered", "job scheduled normally"], 0),
    (["disk read timeout", "kernel reports failure", "reboot required"], 1),
    (["user login accepted", "cache warmup completed", "heartbeat ok"], 0),
    (["io queue stalled", "service unavailable", "node reboot required"], 1),
]


def window_vector(messages):
    return mean_vector([message_vector(message) for message in messages])


train_vectors = [(window_vector(messages), label) for messages, label in train_windows]
test_vectors = [(window_vector(messages), label) for messages, label in test_windows]
print("train windows:", len(train_vectors), "test windows:", len(test_vectors))

## 3. Prototype classifier

정상 prototype과 이상 prototype을 각각 평균 벡터로 만들고, 더 가까운 쪽으로 분류합니다. NeuralLog의 Transformer 분류기보다 훨씬 단순하지만 semantic vector 기반 분류 흐름을 이해하기 좋습니다.

In [ ]:
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


normal_proto = mean_vector([vector for vector, label in train_vectors if label == 0])
anomaly_proto = mean_vector([vector for vector, label in train_vectors if label == 1])


def predict(vector):
    normal_score = cosine(vector, normal_proto)
    anomaly_score = cosine(vector, anomaly_proto)
    return int(anomaly_score > normal_score), anomaly_score - normal_score


predictions = []
for vector, label in test_vectors:
    pred, margin = predict(vector)
    predictions.append(pred)
    print(f"label={label} pred={pred} margin={margin:.3f}")

## 4. Precision, recall, F1 계산

로그 이상 탐지에서는 정상 로그가 훨씬 많을 수 있으므로 accuracy보다 precision, recall, F1을 함께 봐야 합니다.

In [ ]:
def binary_metrics(y_true, y_pred):
    tp = sum(1 for y, p in zip(y_true, y_pred) if y == 1 and p == 1)
    fp = sum(1 for y, p in zip(y_true, y_pred) if y == 0 and p == 1)
    fn = sum(1 for y, p in zip(y_true, y_pred) if y == 1 and p == 0)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


labels = [label for _vector, label in test_vectors]
precision, recall, f1 = binary_metrics(labels, predictions)
print(f"precision={precision:.2f}, recall={recall:.2f}, f1={f1:.2f}")